# chain-rule-elementwise — worked example 1: Backward of out = exp(x) reuses the cached output

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `chain-rule-elementwise`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

For an elementwise forward `out = f(x)`, the Jacobian is diagonal, so backprop is the pointwise product `grad_in = grad_out * f'(x)` — no matmul, no sum. For `out = exp(x)` the local derivative is itself `exp(x)`, which equals the already-cached `out`. Reusing `out` avoids a second `exp` call and keeps `grad_in.shape == x.shape`.

## Worked solution

**Step 1 — identify the forward.** The op is `out = exp(x)`, applied elementwise. Because it acts position-by-position, the full Jacobian `d out / d x` is diagonal, so the chain rule collapses to an elementwise multiply.

**Step 2 — differentiate.** `d/dx exp(x) = exp(x)`. This is the rare case where the derivative equals the function value, so `f'(x) == out`.

**Step 3 — reuse the cache instead of recomputing.** Since the forward pass already produced `out = exp(x)`, we use `out` directly rather than calling `t.exp(x)` again. This is cheaper and avoids a redundant kernel launch.

**Step 4 — apply the chain rule.** `grad_in = grad_out * out`. The shapes line up because every tensor is the same shape as `x`, so the multiply is plain elementwise (no broadcasting needed).

**Step 5 — sanity check against autograd.** We rebuild `x` with `requires_grad=True`, run `exp`, backprop a known `grad_out`, and compare `x.grad` to our hand-written result. They match to floating-point tolerance, confirming the rule.

In [ ]:
def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx exp(x) = exp(x) = out. Reuse the cached forward output.
    return grad_out * out


t.manual_seed(0)
x = t.randn(2, 4, dtype=t.float64)
grad_out = t.randn(2, 4, dtype=t.float64)
out = t.exp(x)
grad_in = exp_back(grad_out, out, x)

# Cross-check against autograd.
xg = x.clone().requires_grad_(True)
t.exp(xg).backward(grad_out)
print("max abs diff vs autograd:", (grad_in - xg.grad).abs().max().item())
print("grad_in shape:", tuple(grad_in.shape))